# MuFu Training

Train MuFu on the five shared seeds (0-4), then look at the losses and compare against RF.


In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

AI_DIR = Path.cwd()
if AI_DIR.name == "notebooks":
    AI_DIR = AI_DIR.parent
elif AI_DIR.name != "ai":
    AI_DIR = AI_DIR / "ai"
sys.path.insert(0, str(AI_DIR / "src"))

from baseline.benchmark import SEEDS, load_benchmark_split, run_benchmark
from baseline.results import RESULTS_PATH, params_json, summarize_seeds
from encoders.smiles_encoder import SmilesEncoder
from mufu.data.dataset import split_datasets
from mufu.experiment import (
    DEFAULT_CONFIG,
    ENCODER,
    benchmark_params,
    make_fit_predict,
)

## Data

Built once and shared by every seed.


In [ ]:
df = load_benchmark_split()
datasets = split_datasets(df)
for split in datasets:
    print(f"{split}: {len(datasets[split])} molecules")

## Training Settings


In [ ]:
# True trains and overwrites this model's rows in results.csv
RUN_TRAINING = False

model_name = "mufu"
seeds = SEEDS  # 0, 1, 2, 3, 4

# override fields here, e.g. lr=5e-4, epochs=30
config = replace(DEFAULT_CONFIG, device="cuda" if torch.cuda.is_available() else "cpu")
print(config)

# one frozen smiles encoder for each seed so embedding cache only filled once
smiles_encoder = SmilesEncoder(freeze=True, max_cache_size=len(df))
params = benchmark_params(config, smiles_encoder)

## Train

Each seed trains a fresh MuFu and saves its best checkpoint and loss log. Results go
to `results.csv` after each seed.


In [ ]:
fit_predict = make_fit_predict(
    datasets,
    smiles_encoder,
    config,
    model_name=model_name,
    checkpoint_dir=AI_DIR / "checkpoints",
    history_dir=AI_DIR / "outputs" / model_name,
    verbose=True,
)

In [ ]:
if RUN_TRAINING:
    summary = run_benchmark(
        fit_predict,
        model=model_name,
        encoder=ENCODER,
        params=params,
        seeds=seeds,
        df=df,
    )
    columns = ["dataset", "endpoint", "split", "metric", "value", "std", "n_seeds"]
    display(summary[columns].round(4))
else:
    print("RUN_TRAINING is False; skipping training.")

## Loss Curves

Read from the saved loss logs, no retraining. Train loss is the batch average; validation
loss is over the full split.


In [ ]:
for seed in seeds:
    loss_path = AI_DIR / "outputs" / model_name / f"seed_{seed}_loss.csv"
    if not loss_path.exists():
        print(f"No loss log for seed {seed} yet.")
        continue
    history = pd.read_csv(loss_path)
    plt.figure(figsize=(8, 4))
    plt.plot(history["epoch"], history["train_loss"], label="Training loss")
    plt.plot(history["epoch"], history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} seed {seed}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Recorded Test Results

MuFu (current settings) and RF from `results.csv`. `value` is the mean, `std` the sample
std (undefined for one seed), and `n_seeds` the number of seeds.


In [ ]:
results = pd.read_csv(RESULTS_PATH)
# only current-settings MuFu rows so older configs don't mix in
is_mufu = results["model"] == model_name
results = results[~is_mufu | (results["params"] == params_json(params))]
comparison = summarize_seeds(results)
comparison = comparison[comparison["split"] == "test"]
columns = ["model", "dataset", "endpoint", "metric", "value", "std", "n_seeds"]
comparison[columns].round(4)

## MuFu vs RF

Test mean ± std across seeds. RMSE: lower is better. R² and ROC-AUC: higher is better.


In [ ]:
if model_name not in comparison["model"].values:
    print(
        f"No test results for {model_name} yet. Run training before comparing with RF."
    )
else:
    for rf_name in ["rf_bioactivity", "rf_toxicity"]:
        rf_results = comparison[comparison["model"] == rf_name]
        for metric in rf_results["metric"].unique():
            endpoints = rf_results.loc[rf_results["metric"] == metric, "endpoint"]
            plot_rows = comparison[
                comparison["model"].isin([rf_name, model_name])
                & (comparison["metric"] == metric)
                & comparison["endpoint"].isin(endpoints)
            ]
            means = plot_rows.pivot(index="endpoint", columns="model", values="value")
            errors = plot_rows.pivot(index="endpoint", columns="model", values="std")
            model_order = [rf_name, model_name]
            means = means[model_order]
            errors = errors[model_order]
            ax = means.plot.barh(
                xerr=errors,
                capsize=3,
                figsize=(10, max(4, 0.45 * len(means))),
            )
            direction = "lower is better" if metric == "rmse" else "higher is better"
            ax.set_xlabel(f"{metric} ({direction})")
            ax.set_ylabel("Endpoint")
            ax.set_title(f"{model_name} vs {rf_name} (test)")
            ax.grid(True, axis="x", alpha=0.3)
            plt.tight_layout()
            plt.show()